# Google Colab Setup for Data Analysis Project

**IMPORTANT: Run the first cell below to set up your environment!**
This works in Google Colab, GitHub Codespaces, and locally.

## Before you run in Colab:
1. Click the **Secrets** icon on the left sidebar.
2. Add these secrets:
   - Name: `OCI_ACCESS_KEY` | Value: `YOUR_ACCESS_KEY`
   - Name: `OCI_SECRET_KEY` | Value: `YOUR_SECRET_KEY`
   - Name: `CENSUS_API_KEY` | Value: `YOUR_CENSUS_API_KEY`
   
*(Make sure "Notebook access" is checked for all secrets).*

In [ ]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import subprocess
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)

print(f"🚀 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Clone repo in Colab if needed
if IS_COLAB and not os.path.exists("computer-data-analysis-report"):
    print("Cloning repository...")
    !git clone --depth 1 https://github.com/Aidas-dev/computer-data-analysis-report.git

# Change to repo directory
repo_dir = "computer-data-analysis-report"
if os.path.exists(repo_dir):
    %cd {repo_dir}
    !git pull -q
else:
    !cd {repo_dir} 2>/dev/null || echo "Repo not found"

# Install uv (faster than pip)
!pip install uv -q

# Install dependencies with uv (bypasses Colab's pip resolver issues)
!uv pip install --system -r requirements.txt -q

# Setup DVC in Colab
if IS_COLAB:
    from google.colab import userdata
    try:
        access_key = userdata.get("OCI_ACCESS_KEY")
        secret_key = userdata.get("OCI_SECRET_KEY")
        os.environ["CENSUS_API_KEY"] = userdata.get("CENSUS_API_KEY") or os.getenv("CENSUS_API_KEY", "")
        !dvc remote modify --local oracle_remote access_key_id {access_key}
        !dvc remote modify --local oracle_remote secret_access_key {secret_key}
        !dvc pull -q
    except:
        pass  # Secrets not configured, skip DVC pull

# Setup Census API key from environment
if not os.getenv("CENSUS_API_KEY"):
    # Try to load from .env file
    from pathlib import Path
    env_file = Path(".env")
    if env_file.exists():
        for line in env_file.read_text().strip().split("\n"):
            if line.startswith("CENSUS_API_KEY="):
                os.environ["CENSUS_API_KEY"] = line.split("=", 1)[1]

print("Census API key loaded:" if os.getenv("CENSUS_API_KEY") else "Census API key: NOT SET")

print("✅ Environment ready!")